# Data Download and Processing

## Project Preamble

This project studies whether a neural network can improve professional macroeconomic forecasts from the ECB Survey of Professional Forecasters (SPF). Instead of forecasting inflation or GDP growth directly, the model learns the SPF forecast error:

```text
forecast_error = actual_value - rolling_1y_forecast
```

The final modelling question is whether the neural network can predict part of this error out of sample and therefore create a corrected forecast that performs better than the raw SPF forecast.

## Data Sources

The project combines three main data sources:

1. **ECB Survey of Professional Forecasters (SPF)**: individual forecaster-level expectations for euro-area HICP inflation and real GDP growth. I manually downloaded the raw SPF individual forecast CSV files and placed them in `Data/SPF_individual_forecasts/`.
2. **EA-MD-QD macro dataset**: a large euro-area macro and financial dataset used as the predictor panel. I manually downloaded the full zipped EA-MD-QD dataset, kept the euro-area files in `Data/EA-MD-QD-2026-04/`, and ran the dataset authors' Python preprocessing routine through `Data/EA-MD-QD-2026-04/RUN.ipynb`. That routine creates the processed quarterly euro-area panel used here: `Data/EA-MD-QD-2026-04/data_TR2/EAdataQM_TR2.xlsx`.
3. **Eurostat realized outcomes**: official realized HICP inflation and real GDP growth. These are downloaded through the Eurostat API inside this notebook and are used to calculate the realized forecast-error target.

## Why Processing Is Needed

The raw SPF files are very close to the original survey format. Each CSV is one survey round, and inside each file several tables are stacked vertically. The HICP table, real GDP table, core inflation table, unemployment table, and assumptions blocks appear in the same file, separated by section headers and blank rows. Forecast targets also appear in different formats: calendar years such as `2020`, HICP rolling targets such as `2020Dec`, and GDP rolling targets such as `2020Q3`. The raw files also contain density-forecast probability bins, while this project only needs the individual point forecasts.

Because of this, the first part of the notebook parses the raw CSV structure, keeps only HICP and real GDP point forecasts, standardizes names and types, classifies the forecast horizon, and reshapes the data into a clean SPF panel.

## What This Notebook Does

This notebook is the combined data pipeline. It first recreates the cleaned SPF datasets and then builds the final modelling dataset:

- reads the manually downloaded raw SPF CSV files
- extracts `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT` from the HICP and real GDP sections
- converts the long SPF file into a wide forecaster-level panel
- keeps the rolling one-year-ahead forecast as the main forecast to be corrected
- loads the processed EA-MD-QD euro-area macro panel
- downloads realized HICP and real GDP outcomes from Eurostat
- merges SPF forecasts, realized outcomes, and lag-aligned macro predictors
- saves the final modelling file as `Data/final_model_dataset.csv`

The final dataset is structured at the forecaster-survey-variable level: one row for each forecaster, survey round, and target variable. It contains identifiers, the SPF rolling one-year forecast, the realized actual value, the forecast-error target, SPF-derived features, past-error features, and EA-MD-QD macro predictors.

## Engineered Variables

From the SPF panel, I engineer consensus and disagreement features across available forecast horizons, including the consensus mean, forecast disagreement, number of forecasters, distance from consensus, and absolute distance from consensus. I also add horizon-shape features such as `near_term_tilt`, `forward_acceleration`, and `curvature`, which summarize how each forecaster's short-run and calendar-year expectations relate to each other.

From the forecast-error history, I add leak-safe past-error variables. These summarize each forecaster's previous errors and the variable-level historical errors, but only after the relevant actual values would plausibly have been known.

From the macro panel, I use the processed EA-MD-QD predictors with a conservative one-quarter lag. I also engineer interpretable macro variables from the raw EA-MD-QD levels: `gdp_yoy_growth`, `hicp_yoy_growth`, and `term_spread`. These help connect the high-dimensional macro panel to the economic concepts in the SPF forecasts.

## Why the Neural Network Is in a Separate Notebook

This notebook only prepares the data. The neural-network training is done in a separate Google Colab notebook, `nn_replication_a100.ipynb`, because the full expanding-window training loop is computationally heavy. I use an A100 GPU runtime there to make the repeated neural-network refits feasible within the assignment timeline.


## Imports and Settings

All shared imports are loaded once at the top so the rest of the notebook can focus on the data-processing stages.


In [1]:
import glob
import math
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

# Resolve paths from the repository root so the notebook works after cloning.
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "Data").exists() and (folder / "README.md").exists():
            return folder
    return start


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "Data"

# Keep preview tables compact enough to scan in the notebook.
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)


## Part 1 - Step 1: Cleaning the ECB Survey of Professional Forecasters (SPF) Data

## Purpose

This notebook reads the raw ECB SPF individual-round CSV files, extracts the
**point forecasts** for **HICP inflation** and **real GDP growth**, and
produces a single clean panel dataset.

The final table has one row per *forecaster × survey round × target period*
and contains the columns we need for the later modelling steps:

| column | meaning |
|---|---|
| `survey_round` | the quarter the survey was conducted (e.g. `2020Q1`) |
| `variable` | `HICP` (inflation) or `RGDP` (real GDP growth) |
| `target_period` | the period the forecast refers to (e.g. `2020`, `2021Dec`, `2020Q3`) |
| `fct_source` | anonymous forecaster ID (integer, stable across rounds) |
| `point_forecast` | the forecaster's point estimate |
| `horizon_type` | label for the forecast horizon: `current_year`, `next_year`, `year_after_next`, `longer_term`, `rolling_1y`, `rolling_2y` |

We keep **all** available calendar-year and rolling horizons at this stage.
Downstream notebooks will filter to the rolling one-year-ahead horizon.

## 1 - Raw SPF Input Files

This stage locates the raw ECB SPF CSV files. Sorted filenames preserve the chronological survey-round order used by the parser below.


In [2]:
# Locate the raw ECB SPF files from the repo-level Data folder.
SPF_DATA_DIR = DATA_ROOT / "SPF_individual_forecasts"
DATA_DIR = str(SPF_DATA_DIR)
csv_files = sorted(glob.glob(str(SPF_DATA_DIR / "*.csv")))


---
## 1 — Raw CSV layout

Each SPF CSV is one survey round with several stacked sections. The parser below only keeps the HICP and real-GDP point-forecast sections, extracts `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`, and ignores density-forecast bins plus unrelated blocks such as core inflation, unemployment, and assumptions.

Rolling targets have month or quarter suffixes, such as `2020Dec` for HICP or `2020Q3` for real GDP. Calendar-year targets are plain years such as `2020`.


---
## 2 — Parsing a single CSV file

Our strategy for parsing one file:

1. Read the file line by line.
2. Detect section headers to know which variable (HICP / RGDP) we are in.
3. When we see a `TARGET_PERIOD` header row, read subsequent data rows
   until we hit a blank line or a new section header.
4. Keep only `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`.
5. Tag every row with the current variable name.

In [3]:
# Regex headers for the two blocks we want. SKIP_PATTERNS marks the blocks we
# ignore (core inflation, unemployment, assumptions) so the parser stops collecting.
SECTION_PATTERNS = {
    "HICP": re.compile(r"^INFLATION EXPECTATIONS;\s*YEAR-ON-YEAR CHANGE IN HICP"),
    "RGDP": re.compile(r"^GROWTH EXPECTATIONS;\s*YEAR-ON-YEAR CHANGE IN REAL GDP"),
}

SKIP_PATTERNS = [
    re.compile(r"^CORE INFLATION"),
    re.compile(r"^EXPECTED UNEMPLOYMENT"),
    re.compile(r"^ASSUMPTIONS"),
]


# Parse one SPF survey-round CSV and keep TARGET_PERIOD, FCT_SOURCE, POINT, and variable.
def parse_spf_file(filepath: str) -> pd.DataFrame:
    rows = []
    current_var = None  # which section we're inside; None = a block we don't use

    with open(filepath, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            matched_var = None
            for var_name, pattern in SECTION_PATTERNS.items():
                if pattern.search(line):
                    matched_var = var_name
                    break

            if matched_var:
                current_var = matched_var
                continue

            if any(pattern.search(line) for pattern in SKIP_PATTERNS):
                current_var = None
                continue

            if line.startswith("TARGET_PERIOD") or line == "" or line.replace(",", "") == "":
                continue

            if current_var is not None:
                fields = line.split(",")
                target_period = fields[0].strip()
                fct_source = fields[1].strip()
                point_raw = fields[2].strip()

                if point_raw == "" or target_period == "" or fct_source == "":
                    continue

                rows.append((target_period, fct_source, point_raw, current_var))

    return pd.DataFrame(rows, columns=["TARGET_PERIOD", "FCT_SOURCE", "POINT", "variable"])


---
## 3 — Parsing all 110 survey rounds

We loop over every CSV in the `Data/SPF_individual_forecasts` folder. The **survey round** (e.g. `2020Q1`) is extracted from the filename and becomes the time index for the later modelling dataset.

In [4]:
frames = []

for fpath in csv_files:
    survey_round = os.path.splitext(os.path.basename(fpath))[0]  # "2020Q1.csv" -> "2020Q1"

    df = parse_spf_file(fpath)
    df["survey_round"] = survey_round
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

---
## 4 — Type conversions and basic cleaning

The parser reads every field as text. Numeric conversion is needed before we can compare forecasts, calculate summaries, or merge reliably. Rows that cannot be converted are dropped because they do not contain usable point forecasts.

In [5]:
# coerce -> bad strings become NaN, so one dropna removes rows without a usable forecast
raw["POINT"] = pd.to_numeric(raw["POINT"], errors="coerce")
raw["FCT_SOURCE"] = pd.to_numeric(raw["FCT_SOURCE"], errors="coerce")

raw = raw.dropna(subset=["POINT", "FCT_SOURCE"]).copy()
raw["FCT_SOURCE"] = raw["FCT_SOURCE"].astype(int)


---
## 5 — Classifying the forecast horizon

This is the most important cleaning step. Each row has a `TARGET_PERIOD`
and a `survey_round`. From these two we can figure out **which forecast
horizon** the row belongs to.

### Horizon classification rules

| horizon label | how to detect it |
|---|---|
| `rolling_1y` | HICP/RGDP target has a month or quarter suffix. Within each `survey_round × variable`, it is the **earliest** rolling target period. |
| `rolling_2y` | Same month/quarter format, but it is the **second** rolling target period within that `survey_round × variable`. |
| `current_year` | Target is a plain year equal to the survey year. |
| `next_year` | Target is a plain year equal to survey year + 1. |
| `year_after_next` | Target is a plain year equal to survey year + 2. |
| `longer_term` | Target is a plain year ≥ survey year + 3. |

The trick is that **rolling horizons always contain a month or quarter
suffix** (like `Dec` or `Q3`), whereas calendar-year horizons are just
a four-digit year. We do **not** classify rolling horizons only from
`target_year - survey_year`, because early rounds can have both `1999Dec`
and `2000Dec` in the same survey. The ECB description says these are the
one-year- and two-year-ahead rolling horizons, respectively.

In [6]:
# Two-step classification: calendar-year targets can be labelled directly from
# target_year - survey_year; rolling targets get ranked within each round further down.
RE_YEAR_ONLY = re.compile(r"^(\d{4})$")
RE_MONTH = re.compile(r"^(\d{4})(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$")
RE_QUARTER = re.compile(r"^(\d{4})Q([1-4])$")


# Convert month/quarter TARGET_PERIOD strings to pandas timestamps for chronological sorting.
def target_period_sort_key(target_period: str) -> pd.Timestamp:
    m_month = RE_MONTH.match(target_period)
    if m_month:
        year, month_name = m_month.groups()
        month_num = pd.to_datetime(month_name, format="%b").month  # "Dec" -> 12
        return pd.Timestamp(int(year), month_num, 1)

    m_quarter = RE_QUARTER.match(target_period)
    if m_quarter:
        year, quarter = m_quarter.groups()
        month_num = (int(quarter) - 1) * 3 + 1  # Q1/Q2/Q3/Q4 -> Jan/Apr/Jul/Oct
        return pd.Timestamp(int(year), month_num, 1)

    return pd.NaT


# Classify plain calendar-year targets; rolling month/quarter targets are assigned below.
def classify_calendar_horizon(target_period: str, survey_round: str) -> str:
    survey_year = int(survey_round[:4])

    if RE_MONTH.match(target_period) or RE_QUARTER.match(target_period):
        return "rolling_unassigned"

    m_year = RE_YEAR_ONLY.match(target_period)
    if m_year:
        target_year = int(m_year.group(1))
        diff = target_year - survey_year
        if diff == 0:
            return "current_year"
        elif diff == 1:
            return "next_year"
        elif diff == 2:
            return "year_after_next"
        else:
            return "longer_term"

    return "unknown"


raw["horizon_type"] = raw.apply(
    lambda r: classify_calendar_horizon(r["TARGET_PERIOD"], r["survey_round"]),
    axis=1,
)

rolling_mask = raw["horizon_type"] == "rolling_unassigned"
rolling_targets = (
    raw.loc[rolling_mask, ["survey_round", "variable", "TARGET_PERIOD"]]
    .drop_duplicates()  # one target label per survey round and variable is enough for ranking
    .assign(target_sort=lambda d: d["TARGET_PERIOD"].apply(target_period_sort_key))
    .sort_values(["survey_round", "variable", "target_sort"])
)
rolling_targets["rolling_rank"] = (
    rolling_targets.groupby(["survey_round", "variable"]).cumcount() + 1  # 1st rolling target -> 1y, 2nd -> 2y
)
rolling_targets["rolling_label"] = rolling_targets["rolling_rank"].map(
    {1: "rolling_1y", 2: "rolling_2y"}
).fillna("rolling_longer")

raw = raw.merge(
    rolling_targets[["survey_round", "variable", "TARGET_PERIOD", "rolling_label"]],
    on=["survey_round", "variable", "TARGET_PERIOD"],
    how="left",
)
raw.loc[rolling_mask, "horizon_type"] = raw.loc[rolling_mask, "rolling_label"]
raw = raw.drop(columns=["rolling_label"])


---
## 6 — Rename columns to final schema

We rename the raw column names to the clean, lowercase names that we
will use throughout the project.

In [7]:
clean = raw.rename(columns={
    "TARGET_PERIOD": "target_period",
    "FCT_SOURCE":    "fct_source",
    "POINT":         "point_forecast",
})

# Reorder columns into a logical sequence
clean = clean[["survey_round", "variable", "target_period",
               "fct_source", "point_forecast", "horizon_type"]]


---
## 7 — Extract the modelling forecast columns

For the wide modelling table, keep the rolling one-year forecast, rolling two-year forecast, next-year forecast, and current-year forecast. The sparse early `rolling_longer` horizon stays in the long archive file but is no longer part of the main modelling pipeline.


In [8]:
# One slice per horizon; the next cell merges them side by side.
def make_horizon_slice(horizon_type, target_col, forecast_col):
    return (
        clean
        .loc[clean["horizon_type"] == horizon_type]
        .rename(columns={"point_forecast": forecast_col, "target_period": target_col})
        [["survey_round", "variable", "fct_source", target_col, forecast_col]]
    )


rolling_1y = make_horizon_slice("rolling_1y", "rolling_1y_target", "rolling_1y_forecast")
rolling_2y = make_horizon_slice("rolling_2y", "rolling_2y_target", "rolling_2y_forecast")
next_year = make_horizon_slice("next_year", "next_year_target", "next_year_forecast")


In [9]:
# Outer merges keep forecasters who reported some horizons but not others.
horizon_frames = [rolling_1y, rolling_2y, next_year]
merged = horizon_frames[0].copy()

for frame in horizon_frames[1:]:
    merged = pd.merge(
        merged,
        frame,
        on=["survey_round", "variable", "fct_source"],
        how="outer",
    )


---
## 8 — Add current-year forecast as an extra feature

The current-year forecast is another useful piece of information.
By the time Q3 or Q4 rolls around, the current-year forecast is nearly
a realized value, so it captures how much information about the
macro economy the forecaster already has. We add it as an additional
column.

In [10]:
current_year = (
    clean
    .loc[clean["horizon_type"] == "current_year"]
    .rename(columns={"point_forecast": "current_year_forecast", "target_period": "current_year_target"})
    [["survey_round", "variable", "fct_source", "current_year_target", "current_year_forecast"]]
)

merged = pd.merge(
    merged,
    current_year,
    on=["survey_round", "variable", "fct_source"],
    how="left",
)


---
## 9 — Parse the survey round into year and quarter columns

Having `survey_year` and `survey_quarter` as separate numeric columns
will be handy for sorting, plotting, and later for creating time-based
features like "survey quarter" dummies.

In [11]:
merged["survey_year"] = merged["survey_round"].str[:4].astype(int)
merged["survey_quarter"] = merged["survey_round"].str[-1].astype(int)


---
## 10 — Also keep the long-form dataset

The wide table keeps the selected modelling horizons as columns. The long table keeps every parsed SPF horizon as rows, which is useful for checks or later extensions.

We save both.


In [12]:
# Add survey_year and survey_quarter to the long-form dataset too.
clean["survey_year"] = clean["survey_round"].str[:4].astype(int)
clean["survey_quarter"] = clean["survey_round"].str[-1].astype(int)


---
## 11 — Save the cleaned datasets to CSV

These two CSVs are the handoff from raw SPF files to the downstream modelling notebooks: the long file preserves every horizon, while the wide file keeps the selected horizons as columns.

In [13]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)

long_path = DATA_ROOT / "spf_clean_long.csv"
wide_path = DATA_ROOT / "spf_clean_wide.csv"

clean.to_csv(long_path, index=False)
merged.to_csv(wide_path, index=False)

pd.DataFrame(
    {
        "file": [str(long_path.relative_to(PROJECT_ROOT)), str(wide_path.relative_to(PROJECT_ROOT))],
        "rows": [len(clean), len(merged)],
        "columns": [clean.shape[1], merged.shape[1]],
    }
)


,file,rows,columns
0,Data/spf_clean_long.csv,61677,8
1,Data/spf_clean_wide.csv,12357,13


### Column descriptions

| column | description |
|---|---|
| `survey_round` | Quarter when the ECB conducted the survey (e.g. `2020Q1`). Extracted from the filename. |
| `variable` | Which macroeconomic variable: `HICP` = year-on-year change in the Harmonised Index of Consumer Prices (inflation); `RGDP` = year-on-year change in real GDP (growth). |
| `fct_source` | Anonymous integer ID for each forecaster. The same number refers to the same institution across all rounds. |
| `rolling_1y_target` | The specific target period for the rolling one-year-ahead forecast (e.g. `2020Dec` for HICP, `2020Q3` for RGDP). |
| `rolling_1y_forecast` | The forecaster's point estimate for the rolling one-year-ahead horizon. This is our **main forecast** column. |
| `rolling_2y_target` | The specific target period for the rolling two-year-ahead forecast. |
| `rolling_2y_forecast` | The forecaster's point estimate for the rolling two-year-ahead horizon. |
| `next_year_target` | The calendar year being forecast as "next year". |
| `next_year_forecast` | The forecaster's point estimate for the next calendar year. |
| `current_year_target` | The calendar year being forecast as "current year". |
| `current_year_forecast` | The forecaster's point estimate for the current calendar year. |
| `survey_year` | Numeric year of the survey (integer). |
| `survey_quarter` | Numeric quarter of the survey (1-4). |


---
## Summary

We started with 110 raw ECB SPF CSV files and parsed the HICP and real-GDP point forecasts.

**What we did:**

1. Identified the HICP and RGDP sections in each survey-round file.
2. Extracted `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`.
3. Converted types and dropped rows without usable point forecasts.
4. Classified each row into calendar-year or rolling forecast horizons.
5. Built a wide modelling table with `rolling_1y_forecast`, `rolling_2y_forecast`, `next_year_forecast`, and `current_year_forecast`.
6. Saved both the long archive table and the wide modelling table.


---
## Part 2 - Step 3: Final Modelling Dataset

The cleaned wide SPF file from Step 1 is now combined with EA-MD-QD predictors and realized Eurostat outcomes to build the final supervised-learning dataset.


This notebook builds the final supervised-learning dataset for the neural network.

The goal is to predict the one-year-ahead SPF forecast error:

```text
forecast_error = realized_actual_value - SPF_rolling_1y_forecast
```

The dataset combines five blocks of information:

- cleaned ECB SPF forecasts from Step 1
- maintained euro-area macro and financial predictors from EA-MD-QD
- target-aligned engineered macro predictors from raw EA-MD-QD levels
- SPF consensus and forecast-horizon shape features
- leak-safe historical forecast-error features

The target construction and date alignment stay explicit because
those choices matter more than the neural-network architecture for whether the
experiment is credible.

## Step 3 Paths and Lag Settings

The final predictor block comes from the maintained EA-MD-QD workbook that was already processed with the package's own transformation and imputation routine. Imports live in the unified setup above; this cell keeps only paths and the conservative macro-data lag.


In [14]:
DATA_DIR = DATA_ROOT

SPF_WIDE_PATH = DATA_DIR / "spf_clean_wide.csv"
EA_MD_QD_PATH = DATA_DIR / "EA-MD-QD-2026-04" / "data_TR2" / "EAdataQM_TR2.xlsx"
EA_MD_QD_RAW_PATH = DATA_DIR / "EA-MD-QD-2026-04" / "EAdata.xlsx"
FINAL_DATASET_PATH = DATA_DIR / "final_model_dataset.csv"

# Use the latest fully completed EA-MD-QD quarter before the SPF survey quarter.
EA_PREDICTOR_LAG_QUARTERS = 1


## Load SPF and EA-MD-QD Inputs

`spf_clean_wide.csv` is one row per forecaster, survey round, and variable.
`EAdataQM_TR2.xlsx` is the processed quarterly EA-MD-QD panel. I also load
`EAdata.xlsx`, the raw EA-MD-QD level file, to create interpretable year-on-year
GDP and HICP momentum variables. EA here means **euro area**, which is the
correct geographic aggregate for the ECB SPF HICP and real GDP questions.

In [15]:
spf = pd.read_csv(SPF_WIDE_PATH)  # Step 1 output: forecaster-round-variable SPF panel

ea_md_qd = pd.read_excel(EA_MD_QD_PATH)  # transformed quarterly macro predictor panel
ea_md_qd_raw = pd.read_excel(EA_MD_QD_RAW_PATH, sheet_name="data")  # raw levels for interpretable YoY features


## Engineer Target-Aligned Macro Predictors

The processed EA-MD-QD file already contains many transformed predictors, but a
few economically important transformations are clearer if we create them
explicitly:

- `gdp_yoy_growth`: year-on-year growth from raw `GDP_EA` levels
- `hicp_yoy_growth`: year-on-year growth from raw `HICPOV_EA` index levels
- `term_spread`: long-term interest rate minus three-month rate

The year-on-year variables are target-aligned because their units match the SPF
inflation and GDP-growth questions. They are still merged with the same
one-quarter lag as the other EA-MD-QD predictors, so they are predictors rather
than realized target values.

In [16]:
ea_raw = ea_md_qd_raw.rename(columns={"Time": "raw_date"}).copy()
ea_raw["raw_date"] = pd.to_datetime(ea_raw["raw_date"])  # standardize dates before quarterly grouping

# Raw levels -> quarterly YoY growth. GDP is already quarterly (take the last value
# per quarter); HICP is a monthly index (use the quarterly mean before the YoY change).
def quarterly_yoy_from_level(raw_frame, value_col, output_col, aggregation="mean"):
    levels = raw_frame[["raw_date", value_col]].copy()
    levels[value_col] = pd.to_numeric(levels[value_col], errors="coerce")
    levels = levels.dropna(subset=[value_col])
    levels["ea_md_qd_date"] = levels["raw_date"].dt.to_period("Q").dt.start_time

    quarterly_level = (
        levels
        .groupby("ea_md_qd_date")[value_col]
        .agg(aggregation)
        .sort_index()
    )
    yoy = quarterly_level.pct_change(4) * 100.0  # vs same quarter last year

    return yoy.rename(output_col).reset_index()


gdp_yoy = quarterly_yoy_from_level(
    ea_raw,
    value_col="GDP_EA",
    output_col="gdp_yoy_growth",
    aggregation="last",
)
hicp_yoy = quarterly_yoy_from_level(
    ea_raw,
    value_col="HICPOV_EA",
    output_col="hicp_yoy_growth",
    aggregation="mean",
)

ea_engineered_macro = pd.DataFrame({
    "ea_md_qd_date": pd.to_datetime(ea_md_qd["Date"]).dt.normalize(),
})
ea_engineered_macro = ea_engineered_macro.merge(
    gdp_yoy,
    on="ea_md_qd_date",
    how="left",
    validate="one_to_one",
)
ea_engineered_macro = ea_engineered_macro.merge(
    hicp_yoy,
    on="ea_md_qd_date",
    how="left",
    validate="one_to_one",
)
ea_engineered_macro["term_spread"] = (
    pd.to_numeric(ea_md_qd["LTIRT_EACC"], errors="coerce")
    - pd.to_numeric(ea_md_qd["IRT3M_EACC"], errors="coerce")
)

ea_engineered_macro_cols = [
    "gdp_yoy_growth",
    "hicp_yoy_growth",
    "term_spread",
]


## Clean Target Labels and Build Survey Dates

Some target-period columns are stored as numeric-looking strings after CSV
export, for example `2000.0`. I convert them back to clean labels before any
merge. I also create a real quarterly date for each survey round.

The date rule is:

- `survey_quarter_start`: the first day of the SPF survey quarter
- `ea_reference_date`: the EA-MD-QD quarter allowed as information

With the default one-quarter lag, a 2012Q1 SPF row uses 2011Q4 EA-MD-QD
predictors. This is intentionally conservative.

In [17]:
# Convert target-period labels like 2000.0 back to "2000".
def clean_target_label(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


target_label_cols = [
    "rolling_1y_target",
    "rolling_2y_target",
    "next_year_target",
    "current_year_target",
]

for col in target_label_cols:
    spf[col] = spf[col].map(clean_target_label)

survey_period = pd.PeriodIndex(
    spf["survey_year"].astype(str) + "Q" + spf["survey_quarter"].astype(str),
    freq="Q",
)

spf["survey_quarter_start"] = survey_period.to_timestamp(how="start")
spf["survey_quarter_end"] = survey_period.to_timestamp(how="end").normalize()
spf["ea_reference_date"] = (
    survey_period - EA_PREDICTOR_LAG_QUARTERS
).to_timestamp(how="start")

## Parse Rolling One-Year Target Dates

The target date is useful for constructing past-error features. A previous SPF
error is only usable once its target period has occurred.

- HICP rolling targets are monthly, for example `2000Dec`
- RGDP rolling targets are quarterly, for example `2000Q4`

In [18]:
MONTH_ABBR_TO_NUM = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}


# Return the end date of the SPF rolling target period.
def parse_spf_target_date(variable, target_label):
    if pd.isna(target_label):
        return pd.NaT

    text = str(target_label)

    if variable == "HICP":
        year = int(text[:4])
        month = MONTH_ABBR_TO_NUM[text[4:]]
        return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)

    if variable == "RGDP":
        return pd.Period(text, freq="Q").to_timestamp(how="end").normalize()

    return pd.NaT


spf["target_date"] = [
    parse_spf_target_date(variable, target)
    for variable, target in zip(spf["variable"], spf["rolling_1y_target"])
]

## Keep Rows with One-Year SPF Forecasts

Step 1 intentionally uses outer merges across forecast horizons. That is useful
for an archive table, but the neural network's target is specifically the
rolling one-year-ahead error. Therefore the final training dataset must keep only
rows where `rolling_1y_forecast` is available.

In [19]:
model = spf.loc[spf["rolling_1y_forecast"].notna()].copy()


## Add SPF Consensus Features Across Horizons

The raw individual forecast is not the only useful SPF signal. The model may also learn from how far a forecaster is from the survey consensus and how dispersed forecasts are in that round.

I compute the same consensus feature family for the selected SPF horizons in the wide file:

- rolling one-year
- rolling two-year
- next calendar year
- current calendar year

For each horizon, the notebook creates the consensus mean, consensus median, forecast disagreement, number of forecasters, distance from consensus, and absolute distance from consensus. Missing values are allowed; the NN notebook imputes them inside each training split.


In [20]:
# Same consensus feature family for every horizon, so one loop instead of five
# near-identical groupby blocks.
spf_horizon_specs = {
    "rolling_1y": "rolling_1y_forecast",
    "rolling_2y": "rolling_2y_forecast",
    "next_year": "next_year_forecast",
    "current_year": "current_year_forecast",
}

consensus_frames = []
spf_consensus_feature_cols = []

for horizon, forecast_col in spf_horizon_specs.items():
    horizon_consensus = (
        model
        .groupby(["survey_round", "variable"], as_index=False)
        .agg(
            **{
                f"consensus_mean_{horizon}": (forecast_col, "mean"),
                f"consensus_median_{horizon}": (forecast_col, "median"),
                f"forecast_disagreement_{horizon}": (forecast_col, "std"),
                f"n_forecasters_{horizon}": (forecast_col, "count"),
            }
        )
    )
    consensus_frames.append(horizon_consensus)

    spf_consensus_feature_cols.extend(
        [
            f"consensus_mean_{horizon}",
            f"consensus_median_{horizon}",
            f"forecast_disagreement_{horizon}",
            f"n_forecasters_{horizon}",
            f"distance_from_consensus_{horizon}",
            f"abs_distance_from_consensus_{horizon}",
        ]
    )

consensus = consensus_frames[0]
for extra_consensus in consensus_frames[1:]:
    consensus = consensus.merge(
        extra_consensus,
        on=["survey_round", "variable"],
        how="left",
        validate="one_to_one",
    )

model = model.merge(
    consensus,
    on=["survey_round", "variable"],
    how="left",
    validate="many_to_one",
)

for horizon, forecast_col in spf_horizon_specs.items():
    disagreement_col = f"forecast_disagreement_{horizon}"
    distance_col = f"distance_from_consensus_{horizon}"
    abs_distance_col = f"abs_distance_from_consensus_{horizon}"
    consensus_mean_col = f"consensus_mean_{horizon}"

    model[disagreement_col] = model[disagreement_col].fillna(0.0)
    model[distance_col] = model[forecast_col] - model[consensus_mean_col]
    model[abs_distance_col] = model[distance_col].abs()


## Add SPF Horizon-Shape Features

The clean baseline NN uses the rolling one-year-ahead SPF forecast as the
forecast being corrected. For an ablation, the current-year and next-year SPF
forecasts are also useful because they reveal the forecaster's expected path at
the survey date.

These derived variables summarize the shape of that SPF path:

- `near_term_tilt`: rolling one-year forecast minus current-year forecast
- `forward_acceleration`: next-year forecast minus current-year forecast
- `curvature`: second difference around the rolling one-year forecast

They are known at the time of the survey, so they are not target leakage. When a
forecaster skipped the current-year or next-year horizon, the engineered feature
is left missing and the NN notebook handles it inside each train split.

In [21]:
# First difference between the 12-month-ahead view and the current-year view.
model["near_term_tilt"] = (
    model["rolling_1y_forecast"] - model["current_year_forecast"]
)

# Broader expected acceleration/deceleration from current year to next year.
model["forward_acceleration"] = (
    model["next_year_forecast"] - model["current_year_forecast"]
)

# Second difference: does the rolling horizon sit above/below the line between
# current-year and next-year expectations?
model["curvature"] = (
    (model["next_year_forecast"] - model["rolling_1y_forecast"])
    - (model["rolling_1y_forecast"] - model["current_year_forecast"])
)

spf_horizon_engineered_cols = [
    "near_term_tilt",
    "forward_acceleration",
    "curvature",
]


## Add Basic Time and Variable Features

`survey_round_index` gives the neural network an ordered time coordinate.
`variable_RGDP` lets a pooled model distinguish inflation rows from real GDP rows.
The NN notebook can still train separate HICP and RGDP models, which is the main
specification we currently prefer.

In [22]:
model["survey_round_index"] = (
    (model["survey_year"] - model["survey_year"].min()) * 4
    + (model["survey_quarter"] - 1)
)
model["variable_RGDP"] = (model["variable"] == "RGDP").astype(int)  # only matters for pooled models

## Download Realized Outcomes from Eurostat

The target variable must use the realized value matching the SPF question. I use
Eurostat for the realized outcomes because it gives the official HICP annual
rate of change and real GDP year-on-year growth series.

EA-MD-QD is used for predictors. Its transformed GDP and HICP columns are useful
signals available to the model, but the supervised-learning label should be built
from the same official realized concepts as the SPF forecast targets.

In [23]:
EUROSTAT_BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"


# Eurostat's API returns JSON-stat: the values come as one flat array, and each
# position encodes all dimension coordinates at once (like row-major array indexing).
# Decoding this took some trial and error - the trick is repeated divmod, walking
# the dimension sizes from the last one backwards.
def decode_jsonstat_position(position, sizes):
    coords = []
    remaining = int(position)
    for size in reversed(sizes):
        coords.append(remaining % size)
        remaining //= size
    return list(reversed(coords))


# Fetch one Eurostat JSON-stat series and return a period-indexed Series.
def fetch_eurostat_series(dataset, params, value_name):
    url = f"{EUROSTAT_BASE_URL}/{dataset}"
    request_params = {"format": "JSON", "lang": "en", **params}
    response = requests.get(url, params=request_params, timeout=90)
    response.raise_for_status()
    payload = response.json()

    dim_ids = payload["id"]
    sizes = payload["size"]
    category_labels = {}
    for dim in dim_ids:
        index_map = payload["dimension"][dim]["category"]["index"]
        category_labels[dim] = {pos: label for label, pos in index_map.items()}

    values = payload["value"]
    if isinstance(values, list):
        items = [(i, value) for i, value in enumerate(values)]
    else:
        items = [(int(i), value) for i, value in values.items()]

    records = []
    for flat_position, value in items:
        if value is None:
            continue
        coords = decode_jsonstat_position(flat_position, sizes)
        labels = {
            dim: category_labels[dim][coord]
            for dim, coord in zip(dim_ids, coords)
        }
        records.append({"period": labels["time"], value_name: float(value)})

    out = (
        pd.DataFrame(records)
        .drop_duplicates("period")
        .set_index("period")
        .sort_index()[value_name]
    )
    out.name = value_name
    return out


MONTH_ABBR = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}


# Convert Eurostat monthly periods like "2000-12" to SPF labels like "2000Dec".
def eurostat_month_to_spf_target(period):
    if "-M" in period:
        year, month = period.split("-M")
    else:
        year, month = period.split("-")
    return f"{year}{MONTH_ABBR[int(month)]}"


# Convert Eurostat quarterly periods like "2000-Q4" to SPF labels like "2000Q4".
def eurostat_quarter_to_spf_target(period):
    return period.replace("-Q", "Q").replace("-", "")

In [24]:
# HICP: annual rate of change, all-items HICP.
# EA19 fills early HICP periods that are needed for leak-safe past-error history.
hicp_common_params = {
    "freq": "M",
    "unit": "RCH_A",
    "coicop": "CP00",
    "sinceTimePeriod": "1998-01",
}

hicp_ea20 = fetch_eurostat_series(
    "prc_hicp_manr",
    {**hicp_common_params, "geo": "EA20"},
    "hicp_ea20",
)
hicp_ea19 = fetch_eurostat_series(
    "prc_hicp_manr",
    {**hicp_common_params, "geo": "EA19"},
    "hicp_ea19",
)

hicp_combined = pd.concat([hicp_ea20, hicp_ea19], axis=1)
hicp_combined["actual_value"] = hicp_combined["hicp_ea20"].combine_first(
    hicp_combined["hicp_ea19"]
)
hicp_combined["actual_source"] = np.where(
    hicp_combined["hicp_ea20"].notna(),
    "Eurostat prc_hicp_manr EA20",
    "Eurostat prc_hicp_manr EA19 fallback",
)
hicp_actuals = (
    hicp_combined
    .loc[hicp_combined["actual_value"].notna(), ["actual_value", "actual_source"]]
    .reset_index()
    .rename(columns={"index": "eurostat_period", "period": "eurostat_period"})
)
hicp_actuals["variable"] = "HICP"
hicp_actuals["rolling_1y_target"] = hicp_actuals["eurostat_period"].map(
    eurostat_month_to_spf_target
)

# Real GDP growth: chain-linked volumes, percentage change from the same
# quarter of the previous year. This matches a year-on-year growth concept.
gdp_actuals = fetch_eurostat_series(
    "namq_10_gdp",
    {
        "freq": "Q",
        "unit": "CLV_PCH_SM",
        "s_adj": "SCA",
        "na_item": "B1GQ",
        "geo": "EA20",
        "sinceTimePeriod": "1998-Q1",
    },
    "actual_value",
).reset_index().rename(columns={"index": "eurostat_period", "period": "eurostat_period"})
gdp_actuals["variable"] = "RGDP"
gdp_actuals["rolling_1y_target"] = gdp_actuals["eurostat_period"].map(
    eurostat_quarter_to_spf_target
)
gdp_actuals["actual_source"] = "Eurostat namq_10_gdp EA20"

actual_cols = ["variable", "rolling_1y_target", "actual_value", "actual_source", "eurostat_period"]
actuals = pd.concat(
    [hicp_actuals[actual_cols], gdp_actuals[actual_cols]],
    ignore_index=True,
).drop_duplicates(["variable", "rolling_1y_target"])


## Create the Neural-Network Target Variables

The main label is `forecast_error`.

```text
forecast_error = actual_value - rolling_1y_forecast
```

This sign convention means a positive NN prediction says the raw SPF forecast
should be revised upward, and a negative prediction says it should be revised
downward.

In [25]:
model = model.merge(
    actuals,
    on=["variable", "rolling_1y_target"],
    how="left",
    validate="many_to_one",
)

model["forecast_error"] = model["actual_value"] - model["rolling_1y_forecast"]
model["abs_forecast_error"] = model["forecast_error"].abs()
model["squared_forecast_error"] = model["forecast_error"] ** 2


## Add Leak-Safe Past Forecast-Error Features

Past prediction errors are a sensible feature because they capture persistent
forecaster bias, for example a forecaster who has repeatedly underpredicted
inflation.

The important rule is that the model must not see future errors. For each row,
I only use errors whose target period ended at least one full quarter before
the current SPF survey quarter. This publication buffer is conservative, but
it avoids relying on outcomes that may not have been released yet. I add two versions:

- forecaster-variable history, such as forecaster 12's previous HICP errors
- variable-level history, such as all previous HICP errors

If no history exists yet, the mean features are set to zero and the count feature
is zero. The count tells the neural network whether the historical mean is based
on real evidence or just the no-history baseline.

In [26]:
# Past-error features without leakage. Worked example: for a 2012Q1 survey the
# reference date is end of 2011Q3 (two-quarter buffer), so only errors whose TARGET
# period ended by Sep 2011 count as known history. The buffer matters: an error is
# only knowable once its target period is over AND the actual has been published.
# Getting merge_asof right here took a few tries - both sides have to be sorted on
# the merge key or rows silently misalign.
def merge_past_error_history(frame, keys, prefix):
    out = frame.copy()

    history = out.loc[
        out["forecast_error"].notna() & out["target_date"].notna(),
        [*keys, "target_date", "forecast_error", "abs_forecast_error"],
    ].copy()

    history = (
        history
        .groupby([*keys, "target_date"], as_index=False)
        .agg(
            error_sum=("forecast_error", "sum"),
            abs_error_sum=("abs_forecast_error", "sum"),
            error_count=("forecast_error", "size"),
        )
        .sort_values([*keys, "target_date"])
    )

    history[f"{prefix}_cum_error_sum"] = history.groupby(keys)["error_sum"].cumsum()
    history[f"{prefix}_cum_abs_error_sum"] = history.groupby(keys)["abs_error_sum"].cumsum()
    history[f"{prefix}_past_error_count"] = history.groupby(keys)["error_count"].cumsum()

    history = history.rename(columns={"target_date": "known_target_date"})
    history = history[
        [
            *keys,
            "known_target_date",
            f"{prefix}_cum_error_sum",
            f"{prefix}_cum_abs_error_sum",
            f"{prefix}_past_error_count",
        ]
    ]

    left = (
        out
        .reset_index()
        .sort_values(["past_error_reference_date", *keys])
    )
    right = history.sort_values(["known_target_date", *keys])

    merged = pd.merge_asof(
        left,
        right,
        by=keys,
        left_on="past_error_reference_date",
        right_on="known_target_date",
        direction="backward",  # select the latest history known by the survey date
    )

    merged = merged.sort_values("index").set_index("index")

    count_col = f"{prefix}_past_error_count"
    mean_col = f"{prefix}_past_error_mean"
    abs_mean_col = f"{prefix}_past_abs_error_mean"

    counts = merged[count_col].fillna(0.0)
    out[count_col] = counts.astype(int).to_numpy()

    out[mean_col] = np.where(
        counts > 0,
        merged[f"{prefix}_cum_error_sum"] / counts,
        0.0,
    )
    out[abs_mean_col] = np.where(
        counts > 0,
        merged[f"{prefix}_cum_abs_error_sum"] / counts,
        0.0,
    )

    return out, [mean_col, abs_mean_col, count_col]


# A target is treated as usable history only after a full-quarter publication buffer.
model_survey_period = pd.PeriodIndex(
    model["survey_year"].astype(str) + "Q" + model["survey_quarter"].astype(str),
    freq="Q",
)
model["past_error_reference_date"] = (
    model_survey_period - 2
).to_timestamp(how="end").normalize()

model, forecaster_error_cols = merge_past_error_history(
    model,
    keys=["fct_source", "variable"],
    prefix="forecaster_variable",
)
model, variable_error_cols = merge_past_error_history(
    model,
    keys=["variable"],
    prefix="variable",
)

past_error_feature_cols = forecaster_error_cols + variable_error_cols

## Align EA-MD-QD Predictors to SPF Survey Rounds

The EA-MD-QD panel is quarterly. I first append the engineered macro predictors
to the transformed EA-MD-QD panel, then merge it to SPF rows by survey round using
`merge_asof`, which means each survey round receives the most recent EA-MD-QD
observation at or before `ea_reference_date`.

This produces a many-to-one merge: many individual SPF forecasts share the same
euro-area macro information for the survey round.

In [27]:
ea_md_qd = ea_md_qd.rename(columns={"Date": "ea_md_qd_date"}).copy()
ea_md_qd["ea_md_qd_date"] = pd.to_datetime(ea_md_qd["ea_md_qd_date"]).dt.normalize()

ea_md_qd = ea_md_qd.merge(
    ea_engineered_macro,
    on="ea_md_qd_date",
    how="left",
    validate="one_to_one",
)

ea_feature_cols = [col for col in ea_md_qd.columns if col != "ea_md_qd_date"]
ea_md_qd[ea_feature_cols] = ea_md_qd[ea_feature_cols].apply(pd.to_numeric, errors="coerce")


survey_ea_keys = (
    model[
        [
            "survey_round",
            "survey_year",
            "survey_quarter",
            "survey_quarter_start",
            "ea_reference_date",
        ]
    ]
    .drop_duplicates("survey_round")
    .sort_values("ea_reference_date")
)

survey_ea = pd.merge_asof(
    survey_ea_keys,
    ea_md_qd.sort_values("ea_md_qd_date"),
    left_on="ea_reference_date",
    right_on="ea_md_qd_date",
    direction="backward",  # latest completed macro quarter allowed by the lag rule
)

model = model.merge(
    survey_ea[["survey_round", "ea_md_qd_date", *ea_feature_cols]],
    on="survey_round",
    how="left",
    validate="many_to_one",
)


## Keep Only Rows That Can Train the Model

A supervised row needs:

- a one-year-ahead SPF forecast
- the realized actual value for the target period
- the constructed forecast-error label
- a successful EA-MD-QD macro predictor match

Recent SPF rows whose target has not yet been realized are correctly excluded
from the training file. Rows before the EA-MD-QD sample starts are also
excluded, because otherwise the NN would learn from rows where the full macro
block is missing.

In [28]:
has_actual_value = model["actual_value"].notna()
has_macro_predictors = model[ea_feature_cols].notna().any(axis=1)

trainable = model.loc[has_actual_value & has_macro_predictors].copy()


## Assemble the Final Dataset Columns

This step is needed because the final CSV has to keep three kinds of columns separate:

- identifiers, such as survey round, variable, forecaster ID, and target date
- target columns, such as `actual_value` and `forecast_error`, which are outcomes rather than model inputs
- candidate feature columns, including SPF features, past-error features, and EA-MD-QD macro predictors

The code below also removes candidate predictors that are more than 95% missing before creating `final_dataset`. The NN notebook later performs train-split imputation and scaling, but this early sparsity filter prevents almost-empty columns from entering the modelling file.


In [29]:
# Three kinds of columns: identifiers (bookkeeping), targets (never model inputs!),
# and candidate features. The sparsity filter below drops near-empty candidates.
identifier_cols = [
    "survey_round",
    "survey_year",
    "survey_quarter",
    "survey_quarter_start",
    "survey_quarter_end",
    "survey_round_index",
    "ea_reference_date",
    "ea_md_qd_date",
    "variable",
    "variable_RGDP",
    "fct_source",
    "rolling_1y_target",
    "target_date",
    "past_error_reference_date",
    "eurostat_period",
    "actual_source",
]

target_cols = [
    "actual_value",
    "forecast_error",
    "abs_forecast_error",
    "squared_forecast_error",
]

spf_feature_cols = [
    "rolling_1y_forecast",
    "rolling_2y_forecast",
    "next_year_forecast",
    "current_year_forecast",
    *spf_horizon_engineered_cols,
    *spf_consensus_feature_cols,
]

feature_cols_before_sparsity_filter = [
    "survey_year",
    "survey_quarter",
    "survey_round_index",
    "variable_RGDP",
    *spf_feature_cols,
    *past_error_feature_cols,
    *ea_feature_cols,
]

SPARSE_FEATURE_MISSING_THRESHOLD = 0.95

feature_missing_share = (
    trainable[feature_cols_before_sparsity_filter]
    .isna()
    .mean()
    .sort_values(ascending=False)
)
sparse_feature_cols = set(
    feature_missing_share[
        feature_missing_share > SPARSE_FEATURE_MISSING_THRESHOLD
    ].index
)

sparse_horizon_names = []
for horizon, forecast_col in spf_horizon_specs.items():
    horizon_missing_share = trainable[forecast_col].isna().mean()
    if horizon_missing_share > SPARSE_FEATURE_MISSING_THRESHOLD:
        sparse_horizon_names.append(horizon)

sparse_horizon_feature_cols = set()
for horizon in sparse_horizon_names:
    forecast_col = spf_horizon_specs[horizon]
    for col in feature_cols_before_sparsity_filter:
        if col == forecast_col or col.endswith(f"_{horizon}"):
            sparse_horizon_feature_cols.add(col)

dropped_sparse_feature_cols = sorted(
    sparse_feature_cols | sparse_horizon_feature_cols
)

candidate_feature_cols = [
    col for col in feature_cols_before_sparsity_filter
    if col not in dropped_sparse_feature_cols
]
selected_feature_cols = [
    col for col in candidate_feature_cols
    if col not in identifier_cols and col not in target_cols
]

final_cols = identifier_cols + target_cols + selected_feature_cols
final_dataset = trainable[final_cols].copy()


## Save the Final Dataset

The saved CSV is the dataset used by the NN notebook. It contains one row per survey round, variable, and forecaster, with:

- the raw SPF one-year forecast
- the realized value and forecast-error target
- SPF consensus features for the selected horizons
- SPF horizon-shape features
- sparse candidate features removed using a 95% missing-value rule
- past forecast-error history features
- lag-aligned EA-MD-QD euro-area macro and financial predictors
- engineered `gdp_yoy_growth`, `hicp_yoy_growth`, and `term_spread` predictors

The NN notebook should still impute and scale features inside each train split, using only training data.


In [30]:
final_dataset.to_csv(FINAL_DATASET_PATH, index=False)

save_summary = pd.Series(
    {
        "path": str(FINAL_DATASET_PATH),
        "rows": final_dataset.shape[0],
        "columns": final_dataset.shape[1],
    },
    name="saved_dataset",
)
save_summary

path       /Users/julianreynolds/Library/Mobile Documents...
rows                                                    9810
columns                                                  178
Name: saved_dataset, dtype: object